# Kidney Perturbation Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 28.01.2026
- Date of Last Modification: 01.02.2026 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries

In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from datasets import load_from_disk

from app.count_decoder import apply_count_decoder
from app.infer import embed_dataset, harmonize_tokenize_embed_pipeline, perturb_dataset
from nichejepa.preprocessors import construct_neighbor_graph

ImportError: cannot import name 'SparseDataset' from 'anndata._core.sparse_dataset' (/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/sparse_dataset.py)

## 1.2 Define Config/Params

In [3]:
# Define paths
#model_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/11012026_190057_836' # cross-tissue model
model_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/01112025_081731_614' # benchmarking model
gene_ensembl_map_file_path = '/nfs/team361/sb75/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl'
kidney_perturbation_folder_path = '/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/kidney_perturbation'
cache_directory_path = f'{kidney_perturbation_folder_path}/cache'
results_directory_path = f'{kidney_perturbation_folder_path}/results'
tokenized_data_directory_path = f'{kidney_perturbation_folder_path}/tokenized_data'

In [4]:
niche_key = "NEMO_neighborhood_annotation"
batch_key = 'Study_slice_id'

emb_layer = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_tokens = None
top_k = None

## 1.3 Run Notebook Setup

In [5]:
os.makedirs(tokenized_data_directory_path, exist_ok=True)
os.makedirs(f'{results_directory_path}/figures', exist_ok=True)

In [6]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [7]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [8]:
sc.settings.figdir = f'{results_directory_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

# 2. Analysis

## 2.1 Load Data

In [ ]:
# Load original raw data
adata = sc.read_h5ad(
    "/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/NEMO_UpdatedTokenisationandEmbedding_November22nd25/Labeltransfer/forshare/nemokidneyxeniumatlas_annotated.h5ad")
adata

In [ ]:
# Check columns
adata.obs.head(1)

In [ ]:
# List samples
# CV*** -> pre-treatment
# DI*** -> post-treatment
adata.obs[['Study_slice_id', 'Data_source']].drop_duplicates()

In [ ]:
# Compute sample-wise neighbor graph, concatenate, and store in adata.obsp['spatial_connectivities']
adata = construct_neighbor_graph(
    adata,
    n_neighs=10,
    radius=None,
    delaunay=False,
    include_self_loop=False,
    compute_neighbor_counts=False,
    batch_key=batch_key
)

# Compute raw neighborhood counts
adata.layers['counts_neighborhood'] = (
    adata.obsp['spatial_connectivities'].T @ adata.layers['counts'])
print(adata.layers['counts_neighborhood'][:30].toarray())
adata.layers['counts_neighborhood'] += adata.layers['counts'] # add counts from the index cell
print(adata.layers['counts_neighborhood'][:30].toarray())

# Compute lognorm neighborhood counts (performing same normalization as on raw counts)
adata.X = adata.layers["counts_neighborhood"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm_neighborhood"] = adata.X.copy()
adata.X = adata.layers["lognorm"].copy()

In [ ]:
# Validate that everything is ok
print(adata.obsp['spatial_connectivities'].sum(1))
n = 30
print(adata.layers['counts_neighborhood'][:30].toarray())
print(adata.layers['lognorm_neighborhood'][:30].toarray())
print(adata.layers['counts'][:30].toarray())
print(adata.layers['lognorm'][:30].toarray())
print(adata.X[:30].toarray())

In [ ]:
# Add ensembl IDs to AnnData object
with open(gene_ensembl_map_file_path, "rb") as f:
    gene_ensembl_map_dict = pickle.load(f)

gene_names = [gene_name for gene_name in adata.var_names.tolist()]
adata.var.index = gene_names

harmonized_gene_names = []
matching_ensembl_ids = []
for gene_name in gene_names:
    if gene_name in gene_ensembl_map_dict.keys():
        harmonized_gene_names.append(gene_name)
        matching_ensembl_ids.append(gene_ensembl_map_dict[gene_name])

adata.var['ensembl_id'] = matching_ensembl_ids
adata.var

In [ ]:
# Write AnnData to disk
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

In [ ]:
# Load processed AnnData
adata = sc.read_h5ad(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

## 2.2 Compute Differential Genes Between Pre- and Post-treatment Conditions

Check https://docs.google.com/spreadsheets/d/1i6p3QRTE5Lh32AejWE6Xg2Sye6p3yP7vDdrCMDbLj_I/edit?gid=1921994032#gid=1921994032 for perturbation cases to be tested.

### 2.2.1 Perturbation Case 1

In [ ]:
sid = adata.obs["Study_slice_id"].astype(str)

adata.obs["DI12_vs_CV"] = np.nan
adata.obs.loc[sid.eq("DI12-KID-0-FT-4"), "DI12_vs_CV"] = "DI12"
adata.obs.loc[sid.str.startswith("CV"), "DI12_vs_CV"] = "CV"

adata.obs["DI12_vs_CV"] = adata.obs["DI12_vs_CV"].astype("category")
adata.obs["DI12_vs_CV"].value_counts(dropna=False)

In [ ]:
# Compute differential genes for cell counts (across all cells -> global)
sc.tl.rank_genes_groups(
    adata,
    groupby="DI12_vs_CV",
    groups=["DI12"],
    reference="CV",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    key_added="rank_genes_groups_case1_global_pre_post_cell_counts",
)

# Retrieve differential genes for cell counts
n = 50
diff_genes_df = sc.get.rank_genes_groups_df(adata, group="DI12", key="rank_genes_groups_case1_global_pre_post_cell_counts")
diff_genes_df_top = diff_genes_df.head(n)
diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
display(diff_genes_df_top)
diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_global_pre_post_cell_counts.csv")

In [ ]:
# Compute differential genes for cell counts (across each niche)
# Loop over niches and run DE within each
min_cells_per_group = 20

summary_rows = []

# Get categories (handles categorical + non-categorical)
cats = adata.obs[niche_key].astype("category").cat.categories

for cat in cats:
    # Cells in this niche AND in either DI12 or CV
    mask = (adata.obs[niche_key] == cat) & (adata.obs["DI12_vs_CV"].isin(["DI12", "CV"]))
    n_total = int(mask.sum())
    if n_total == 0:
        summary_rows.append({"niche": str(cat), "status": "skipped (no DI12/CV cells)", "n_DI12": 0, "n_CV": 0})
        continue

    sub = adata[mask].copy() # copy so we can compute safely without view warnings

    counts = sub.obs["DI12_vs_CV"].value_counts()
    n_di12 = int(counts.get("DI12", 0))
    n_cv   = int(counts.get("CV", 0))

    if n_di12 < min_cells_per_group or n_cv < min_cells_per_group:
        summary_rows.append({
            "niche": str(cat),
            "status": f"skipped (too few cells; min={min_cells_per_group})",
            "n_DI12": n_di12,
            "n_CV": n_cv
        })
        continue

    key = f"rank_genes_groups_case1_niche={cat}_pre_post_cell_counts"

    sc.tl.rank_genes_groups(
        sub,
        groupby="DI12_vs_CV",
        groups=["DI12"],
        reference="CV",
        method="wilcoxon",
        pts=True,
        use_raw=False,
        key_added=key,
    )

    # Retrieve differential genes for cell counts
    n = 50
    diff_genes_df = sc.get.rank_genes_groups_df(sub, group="DI12", key=key)
    diff_genes_df_top = diff_genes_df.head(n)
    diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
    display(diff_genes_df_top)
    diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_niche_{cat}_pre_post_cell_counts.csv")
    
    # Store results on the original adata
    adata.uns[key] = sub.uns[key]

    summary_rows.append({
        "niche": str(cat),
        "status": "ok",
        "n_DI12": n_di12,
        "n_CV": n_cv,
        "uns_key": key
    })

# Store a summary table on the AnnData for bookkeeping
adata.uns["DE_case1_pre_post_cell_counts_summary_by_niche"] = pd.DataFrame(summary_rows)
adata.uns["DE_case1_pre_post_cell_counts_summary_by_niche"]

In [ ]:
# Compute differential genes for neighborhood counts
adata.X = adata.layers['lognorm_neighborhood'].copy()
sc.tl.rank_genes_groups(
    adata,
    groupby="DI12_vs_CV",
    groups=["DI12"],
    reference="CV",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    key_added="rank_genes_groups_case1_global_pre_post_neighborhood_counts",
)

# Retrieve differential genes for neighborhood counts
n = 50
diff_genes_df = sc.get.rank_genes_groups_df(adata, group="DI12", key="rank_genes_groups_case1_global_pre_post_neighborhood_counts")
diff_genes_df_top = diff_genes_df.head(n)
diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
display(diff_genes_df_top)
diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_pre_post_neighborhood_counts.csv")

In [ ]:
# Compute differential genes for neighborhood counts (across each niche)
# Loop over niches and run DE within each
min_cells_per_group = 20

summary_rows = []

# Get categories (handles categorical + non-categorical)
cats = adata.obs[niche_key].astype("category").cat.categories

for cat in cats:
    # Cells in this niche AND in either DI12 or CV
    mask = (adata.obs[niche_key] == cat) & (adata.obs["DI12_vs_CV"].isin(["DI12", "CV"]))
    n_total = int(mask.sum())
    if n_total == 0:
        summary_rows.append({"niche": str(cat), "status": "skipped (no DI12/CV cells)", "n_DI12": 0, "n_CV": 0})
        continue

    sub = adata[mask].copy() # copy so we can compute safely without view warnings

    counts = sub.obs["DI12_vs_CV"].value_counts()
    n_di12 = int(counts.get("DI12", 0))
    n_cv   = int(counts.get("CV", 0))

    if n_di12 < min_cells_per_group or n_cv < min_cells_per_group:
        summary_rows.append({
            "niche": str(cat),
            "status": f"skipped (too few cells; min={min_cells_per_group})",
            "n_DI12": n_di12,
            "n_CV": n_cv
        })
        continue

    key = f"rank_genes_groups_case1_niche={cat}_pre_post_neighborhood_counts"

    sc.tl.rank_genes_groups(
        sub,
        groupby="DI12_vs_CV",
        groups=["DI12"],
        reference="CV",
        method="wilcoxon",
        pts=True,
        use_raw=False,
        key_added=key,
    )

    # Retrieve differential genes for cell counts
    n = 50
    diff_genes_df = sc.get.rank_genes_groups_df(sub, group="DI12", key=key)
    diff_genes_df_top = diff_genes_df.head(n)
    diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
    display(diff_genes_df_top)
    diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case1_niche_{cat}_pre_post_neighborhood_counts.csv")
    
    # Store results on the original adata
    adata.uns[key] = sub.uns[key]

    summary_rows.append({
        "niche": str(cat),
        "status": "ok",
        "n_DI12": n_di12,
        "n_CV": n_cv,
        "uns_key": key
    })

# Store a summary table on the AnnData for bookkeeping
adata.uns["DE_case1_pre_post_neighborhood_counts_summary_by_niche"] = pd.DataFrame(summary_rows)
adata.uns["DE_case1_pre_post_neighborhood_counts_summary_by_niche"]

In [ ]:
adata.X = adata.layers['lognorm'].copy()

In [ ]:
# Write AnnData to disk
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

### 2.2.2 Perturbation Case 2

In [ ]:
# Load processed AnnData
adata = sc.read_h5ad(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

In [ ]:
sid = adata.obs["Study_slice_id"].astype(str)

adata.obs["DI10_vs_CV"] = np.nan
adata.obs.loc[sid.eq("DI10-KID-0-FT-5"), "DI10_vs_CV"] = "DI10"
adata.obs.loc[sid.str.startswith("CV"), "DI10_vs_CV"] = "CV"

adata.obs["DI10_vs_CV"] = adata.obs["DI10_vs_CV"].astype("category")
adata.obs["DI10_vs_CV"].value_counts(dropna=False)

In [ ]:
# Compute differential genes for cell counts (across all cells -> global)
sc.tl.rank_genes_groups(
    adata,
    groupby="DI10_vs_CV",
    groups=["DI10"],
    reference="CV",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    key_added="rank_genes_groups_case2_global_pre_post_cell_counts",
)

# Retrieve differential genes for cell counts
n = 50
diff_genes_df = sc.get.rank_genes_groups_df(adata, group="DI10", key="rank_genes_groups_case2_global_pre_post_cell_counts")
diff_genes_df_top = diff_genes_df.head(n)
diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
display(diff_genes_df_top)
diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case2_global_pre_post_cell_counts.csv")

In [ ]:
# Compute differential genes for cell counts (across each niche)
# Loop over niches and run DE within each
min_cells_per_group = 20

summary_rows = []

# Get categories (handles categorical + non-categorical)
cats = adata.obs[niche_key].astype("category").cat.categories

for cat in cats:
    # Cells in this niche AND in either DI10 or CV
    mask = (adata.obs[niche_key] == cat) & (adata.obs["DI10_vs_CV"].isin(["DI10", "CV"]))
    n_total = int(mask.sum())
    if n_total == 0:
        summary_rows.append({"niche": str(cat), "status": "skipped (no DI10/CV cells)", "n_DI10": 0, "n_CV": 0})
        continue

    sub = adata[mask].copy() # copy so we can compute safely without view warnings

    counts = sub.obs["DI10_vs_CV"].value_counts()
    n_di10 = int(counts.get("DI10", 0))
    n_cv   = int(counts.get("CV", 0))

    if n_di10 < min_cells_per_group or n_cv < min_cells_per_group:
        summary_rows.append({
            "niche": str(cat),
            "status": f"skipped (too few cells; min={min_cells_per_group})",
            "n_DI10": n_di10,
            "n_CV": n_cv
        })
        continue

    key = f"rank_genes_groups_case2_niche={cat}_pre_post_cell_counts"

    sc.tl.rank_genes_groups(
        sub,
        groupby="DI10_vs_CV",
        groups=["DI10"],
        reference="CV",
        method="wilcoxon",
        pts=True,
        use_raw=False,
        key_added=key,
    )

    # Retrieve differential genes for cell counts
    n = 50
    diff_genes_df = sc.get.rank_genes_groups_df(sub, group="DI10", key=key)
    diff_genes_df_top = diff_genes_df.head(n)
    diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
    display(diff_genes_df_top)
    diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case2_niche_{cat}_pre_post_cell_counts.csv")
    
    # Store results on the original adata
    adata.uns[key] = sub.uns[key]

    summary_rows.append({
        "niche": str(cat),
        "status": "ok",
        "n_DI10": n_di10,
        "n_CV": n_cv,
        "uns_key": key
    })

# Store a summary table on the AnnData for bookkeeping
adata.uns["DE_case2_pre_post_cell_counts_summary_by_niche"] = pd.DataFrame(summary_rows)
adata.uns["DE_case2_pre_post_cell_counts_summary_by_niche"]

In [ ]:
# Compute differential genes for neighborhood counts (across all cells -> global)
adata.X = adata.layers['lognorm_neighborhood'].copy()
sc.tl.rank_genes_groups(
    adata,
    groupby="DI10_vs_CV",
    groups=["DI10"],
    reference="CV",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    key_added="rank_genes_groups_case2_global_pre_post_neighborhood_counts",
)

# Retrieve differential genes for neighborhood counts
n = 50
diff_genes_df = sc.get.rank_genes_groups_df(adata, group="DI10", key="rank_genes_groups_case2_global_pre_post_neighborhood_counts")
diff_genes_df_top = diff_genes_df.head(n)
diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
display(diff_genes_df_top)
diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case2_global_pre_post_neighborhood_counts.csv")

In [ ]:
# Compute differential genes for neighborhood counts (across each niche)
# Loop over niches and run DE within each
min_cells_per_group = 20

summary_rows = []

# Get categories (handles categorical + non-categorical)
cats = adata.obs[niche_key].astype("category").cat.categories

for cat in cats:
    # Cells in this niche AND in either DI10 or CV
    mask = (adata.obs[niche_key] == cat) & (adata.obs["DI10_vs_CV"].isin(["DI10", "CV"]))
    n_total = int(mask.sum())
    if n_total == 0:
        summary_rows.append({"niche": str(cat), "status": "skipped (no DI10/CV cells)", "n_DI10": 0, "n_CV": 0})
        continue

    sub = adata[mask].copy() # copy so we can compute safely without view warnings

    counts = sub.obs["DI10_vs_CV"].value_counts()
    n_di10 = int(counts.get("DI10", 0))
    n_cv   = int(counts.get("CV", 0))

    if n_di10 < min_cells_per_group or n_cv < min_cells_per_group:
        summary_rows.append({
            "niche": str(cat),
            "status": f"skipped (too few cells; min={min_cells_per_group})",
            "n_DI10": n_di10,
            "n_CV": n_cv
        })
        continue

    key = f"rank_genes_groups_case2_niche={cat}_pre_post_neighborhood_counts"

    sc.tl.rank_genes_groups(
        sub,
        groupby="DI10_vs_CV",
        groups=["DI10"],
        reference="CV",
        method="wilcoxon",
        pts=True,
        use_raw=False,
        key_added=key,
    )

    # Retrieve differential genes for neighborhood counts
    n = 50
    diff_genes_df = sc.get.rank_genes_groups_df(sub, group="DI10", key=key)
    diff_genes_df_top = diff_genes_df.head(n)
    diff_genes_df_top[["names", "logfoldchanges", "pvals_adj", "scores"]]
    display(diff_genes_df_top)
    diff_genes_df_top.to_csv(f"{results_directory_path}/nemokidneyxeniumatlas_top_{n}_diff_genes_case2_niche_{cat}_pre_post_neighborhood_counts.csv")
    
    # Store results on the original adata
    adata.uns[key] = sub.uns[key]

    summary_rows.append({
        "niche": str(cat),
        "status": "ok",
        "n_DI10": n_di10,
        "n_CV": n_cv,
        "uns_key": key
    })

# Store a summary table on the AnnData for bookkeeping
adata.uns["DE_case2_pre_post_neighborhood_counts_summary_by_niche"] = pd.DataFrame(summary_rows)
adata.uns["DE_case2_pre_post_neighborhood_counts_summary_by_niche"]

In [ ]:
adata.X = adata.layers['lognorm'].copy()

In [ ]:
# Write AnnData to disk
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

### 2.2.3 Perturbation Case 3

Same as in perturbation case 2.

## 2.3 Tokenize Data & Retrieve Embeddings

In [13]:
adata = sc.read_h5ad(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_annotated_processed.h5ad")

In [14]:
adata.X = adata.layers['counts'].copy()

In [17]:
adata = harmonize_tokenize_embed_pipeline(
    adata=adata,
    sample_key=batch_key,
    model_folder_path=model_folder_path,
    cache_directory_path=cache_directory_path,
    gene_perturb_df=None,               
    nproc=4,
    processing_mode='parallel',
    save_dataset_path=f"{tokenized_data_directory_path}/nemokidneyxeniumatlas",
    num_shards=32,
    emb_layer=None,
    agg_excluded_tokens=None,
    top_k=None,
    batch_size=128,
    pin_memory=False,
    num_workers=12,
    use_generator=True,
    add_neigh_cell_ids=True,
    min_cells_per_gene=0,
    ignore_spc_tokens=True)

Start processing sample: CV9-KID-0-FT-5-S4...
Harmonizing sample CV9-KID-0-FT-5-S4...
STEP 1: DATA VALIDATION...
Checking that adata.X contains raw counts...
STEP 2: ADDING ENSEMBL IDS...
Adding ensembl IDs from gene_mapping_dict with file path `/lustre/scratch126/cellgen/lotfollahi/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl`...


/nfs/team361/sb75/nichejepa/src/app/infer.py:771: ImplicitModificationWarning: Trying to modify index of attribute `.var` of view, initializing view as actual.
  adata.var.index = adata_gene_names


Number of genes with matching ensembl IDs: 4949.
Number of genes skipped due to non-matching ensembl IDs: 0.
Filtering genes that have not occurred enough during pretraining...
Number of genes skipped due to not enough pretraining occurrences: 0.
STEP 3: BASIC QUALITY CONTROL...
Filtering cells with less than 10 genes.
Filtering genes expressed in less than 0 cells.


INFO:nichejepa.tokenizers.cell_tokenizers:Loading token dictionary from /nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/01112025_081731_614/token_dictionary.pkl.


STEP 4: ADDING SPECIAL VALUES...
Harmonized sample CV9-KID-0-FT-5-S4.
Tokenizing sample CV9-KID-0-FT-5-S4...
STEP 1: LOADING CONFIG...
STEP 2: TOKENIZING ANNDATA OBJECT...
Filtering cells...
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood...
Normalizing gene expression counts...
Retrieving gene tokens.


/nfs/team361/sb75/nichejepa/src/nichejepa/tokenizers/cell_tokenizers.py:663: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var['ensembl_id'][coding_miRNA_idx]


... (120 outputs left out)

['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr']


0it [00:00, ?it/s]/nfs/team361/sb75/nichejepa/src/app/infer.py:1119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
11859it [4:14:28,  1.29s/it]


Embedded tokenized data.


/nfs/team361/sb75/nichejepa/src/app/infer.py:1541: ImplicitModificationWarning: Setting element `.obsm['cell_emb']` of view, initializing view as actual.
  adata.obsm[key] = values


In [18]:
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_embedded_{model_folder_path.split('/')[-1]}.h5ad")

## 2.4 Perform Perturbations

In [20]:
adata = sc.read_h5ad(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_embedded_{model_folder_path.split('/')[-1]}.h5ad")

In [23]:
dataset = load_from_disk(f"{tokenized_data_directory_path}/nemokidneyxeniumatlas")

### 2.4.1 Perturbation Case 1

Knockout Perturbation:
- Cell & Neighborhood
    - CTLA4 (ENSG00000163599)
    - PDCD1 (ENSG00000188389)

In [25]:
# Define perturbations
perturbed_cell_id = ['all', 'all', 'all', 'all']
perturbed_ensembl_id = ['ENSG00000163599', 'ENSG00000163599', 'ENSG00000188389', 'ENSG00000188389'] # ['all'] * len(perturbed_cell_id)
perturbation_target = ['cell', 'neighborhood', 'cell', 'neighborhood'] # ['neighborhood'] * len(perturbed_cell_id)
perturbation_type = ['knockout', 'knockout', 'knockout', 'knockout'] # ['knockout'] * len(perturbed_cell_id)
foldchange = [np.nan, np.nan, np.nan, np.nan] # [np.nan] * len(perturbed_cell_id)

perturb_df =  pd.DataFrame({
    'perturbed_cell_id': perturbed_cell_id,
    'perturbed_ensembl_id': perturbed_ensembl_id,
    'perturbation_target': perturbation_target,
    'perturbation_type': perturbation_type, 
    'foldchange': foldchange
})
perturb_df

,perturbed_cell_id,perturbed_ensembl_id,perturbation_target,perturbation_type,foldchange
0,all,ENSG00000163599,cell,knockout,NaN
1,all,ENSG00000163599,neighborhood,knockout,NaN
2,all,ENSG00000188389,cell,knockout,NaN
3,all,ENSG00000188389,neighborhood,knockout,NaN


In [216]:
# Run perturbation
perturbed_dataset = perturb_dataset(
    dataset=dataset,
    perturb_df=perturb_df,
    nproc=1,
    model_folder_path=model_folder_path,
    return_only_perturbed_cells=False)
perturbed_dataset

Applying perturbations using dataframe:
  perturbed_cell_id perturbed_ensembl_id perturbation_target  \
0               all      ENSG00000163599                cell   
1               all      ENSG00000163599        neighborhood   
2               all      ENSG00000188389                cell   
3               all      ENSG00000188389        neighborhood   

  perturbation_type  foldchange perturbed_gene_token  
0          knockout         NaN                10985  
1          knockout         NaN                10985  
2          knockout         NaN                15995  
3          knockout         NaN                15995  


Map: 3035788 examples [04:24, 5735.04 examples/s]                                                 


In [231]:
pert_idx_count_dict = {}

for idx, row in perturb_df.iterrows():
    pert_idx_count_dict[f"idx_{idx}"] = sum(f.item() for f in perturbed_dataset[f'pert_flag_idx_{idx}'])
    print(f"{pert_idx_count_dict[f'idx_{idx}']}/{len(perturbed_dataset)} cells are impacted by perturbation {idx}.")

6661/1517894 cells are impacted by perturbation 0.
58472/1517894 cells are impacted by perturbation 1.
3592/1517894 cells are impacted by perturbation 2.
33960/1517894 cells are impacted by perturbation 3.


In [232]:
perturbed_dataset.save_to_disk(f"{tokenized_data_directory_path}/nemokidneyxeniumatlas_perturbed_case1")

Saving the dataset (138/138 shards): 100%|█████| 1517894/1517894 [07:15<00:00, 3483.58 examples/s]


In [108]:
import torch
torch.set_printoptions(threshold=float('inf'))

In [195]:
batch = test_dataset

In [196]:
cell_perturbation = False
seq_len_cell = 256

token_slice = (
    batch["gene_tokens"][:, :seq_len_cell] if cell_perturbation
    else batch["gene_tokens"][:, seq_len_cell:])

TypeError: '>' not supported between instances of 'int' and 'NoneType'

In [ ]:
10985  
15995

In [130]:
token_slice.shape

torch.Size([2, 2560])

In [131]:
offset = 0 if cell_perturbation else seq_len_cell

In [133]:
perturb_df

,perturbed_cell_id,perturbed_ensembl_id,perturbation_target,perturbation_type,foldchange
0,all,ENSG00000163599,cell,knockout,NaN
1,all,ENSG00000163599,neighborhood,knockout,NaN
2,all,ENSG00000188389,cell,knockout,NaN
3,all,ENSG00000188389,neighborhood,knockout,NaN


In [170]:
cell_pert_idx, rel_gene_pert_idx = torch.nonzero(
    token_slice == 15995, as_tuple=True)

In [171]:
cell_pert_idx

tensor([], dtype=torch.int64)

In [172]:
rel_gene_pert_idx

tensor([], dtype=torch.int64)

In [173]:
test_dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 53
})

In [142]:
token_slice[0, 256]

tensor(5421)

In [178]:
test_dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 294987
})

In [233]:
test_dataset = perturbed_dataset.select(range((2304*128+75), (2304*128+76)))

In [234]:
test_dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens', 'pert_flag_idx_0', 'pert_flag_idx_1', 'pert_flag_idx_2', 'pert_flag_idx_3'],
    num_rows: 1
})

In [ ]:
# Retrieve embeddings for perturbed data
perturbed_output_embed = embed_dataset(
    dataset=perturbed_dataset,
    model_folder_path=model_folder_path,
    emb_layer=None,
    agg_excluded_tokens=None,
    top_k=None,
    batch_size=128,
    pin_memory=False,
    num_workers=12,
    include_spatial_cell_emb=True,
    return_token_embeddings=False,
    ignore_spc_tokens=True)

adata.obsm['cell_emb_perturb_case1'] = perturbed_output_embed['cell_emb']
adata.obsm['neighborhood_emb_perturb_case1'] = perturbed_output_embed['neighborhood_emb']
adata.obsm['spatial_cell_emb_perturb_case1'] = perturbed_output_embed['spatial_cell_emb']

STEP 1: LOADING CONFIG...
STEP 2: GENERATING EMBEDDINGS...


INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
    (pos_embed): Embedding(2817, 384, padd

Special token pad ratio 0.0


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
INFO:root:Dataloader and -sampler created.
INFO:root:Loaded pretrained target encoder from epoch 3 with msg: <All keys matched successfully>.
INFO:root:Finished loading checkpoint with read path: /nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/01112025_081731_614/model_checkpoint.pt.


['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr']


4it [00:06,  1.47s/it]

In [ ]:
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_embedded_{model_folder_path.split('/')[-1]}.h5ad")

## 2.5 Decode Counts

In [ ]:
import anndata as ad
from app.count_decoder import apply_count_decoder
from app.count_decoder import _spot_level_metrics, _apply_metric_transform, _to_dense
import numpy as np

adata_path = "/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/kidney_perturbation/nemokidneyxeniumatlas_annotated_processed.h5ad"
checkpoint_path = "results/zinb_decoder.pt"  # or a folder with zinb_decoder.pt inside
emb_key = "cell_emb"
checkpoint_path = "results_neb/zinb_decoder.pt"  # or a folder with zinb_decoder.pt inside
emb_key="neighborhood_emb"

adata = ad.read_h5ad(adata_path)

adata = apply_count_decoder(
    adata,
    emb_key=emb_key,
    model_folder_path=checkpoint_path,  # can be folder or exact file
    decoded_counts_layer_key="decoded_counts",
    # optional overrides:
    # loss_type="nb_softplus",
    # device=0,
    # batch_size=1024,
)
 
preds = _to_dense(adata.layers["decoded_counts"])
target = _to_dense(adata.layers["counts"])
target = _to_dense(adata.layers["counts_neighborhood"])

# optional: same transform used in training metrics
preds = _apply_metric_transform(preds, "none")  # or "log1p"
target = _apply_metric_transform(target, "none")

spot_metrics = _spot_level_metrics(preds, target)
print(spot_metrics)

adata.write_h5ad("/path/to/output_with_decoded_counts.h5ad")

In [45]:
adata = apply_count_decoder(
    adata,
    emb_key='neighborhood_emb',
    model_folder_path='/nfs/team361/mv10/nemo_v6/nichejepa/results_neb/zinb_decoder.pt',
    decoded_counts_layer_key='dec_counts_neighborhood')

NameError: name 'apply_count_decoder' is not defined

In [ ]:
adata = apply_count_decoder(
    adata,
    emb_key='cell_emb',
    model_folder_path='/nfs/team361/mv10/nemo_v6/nichejepa/results/zinb_decoder.pt',
    decoded_counts_layer_key='dec_counts_cell'

In [ ]:
adata = apply_count_decoder(
    adata,
    emb_key='neighborhood_emb_perturb_case1',
    model_folder_path='/nfs/team361/mv10/nemo_v6/nichejepa/results_neb/zinb_decoder.pt',
    decoded_counts_layer_key='dec_counts_perturbed_neighborhood_case1'

In [ ]:
adata = apply_count_decoder(
    adata,
    emb_key='cell_emb_perturb_case1',
    model_folder_path='/nfs/team361/mv10/nemo_v6/nichejepa/results/zinb_decoder.pt',
    decoded_counts_layer_key='dec_counts_perturbed_cell_case1'

In [ ]:
adata.write(f"{kidney_perturbation_folder_path}/nemokidneyxeniumatlas_embedded_{model_folder_path.split('/')[-1]}.h5ad")

In [ ]:
diff testing dec_counts_neighborhood <-> dec_counts_perturbed_neighborhood

In [ ]:
# adata.obsm[emb_key] -> adata.layers[decoded_counts_layer_key]

In [ ]:
perturb_df = perturb_df.head(3)

In [ ]:
perturbed_output_embed = embed_dataset(
    dataset=perturbed_dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    cell_gene_ids=cell_gene_ids,
    neighborhood_gene_ids=neighborhood_gene_ids,
    agg_excluded_tokens=agg_excluded_tokens,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers)

In [ ]:
adata.obsm['perturbed_cell_emb'] = perturbed_output_embed['cell_emb']
adata.obsm['perturbed_neighborhood_emb'] = perturbed_output_embed['neighborhood_emb']

In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_neighborhood_emb',
                key_added='perturbed_neighborhood')
sc.tl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           key_added='perturbed_neighborhood_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_neighborhood_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_cell_emb',
                key_added='perturbed_cell')
sc.tl.umap(adata,
           neighbors_key='perturbed_cell',
           key_added='perturbed_cell_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_cell_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_cell',
           color=label)